In [5]:
import pandas as pd

In [7]:
df=pd.read_csv('scaled_fraud_training_data_100k.csv')

In [8]:
df

,transaction_id,customer_id,beneficiary_id,merchant_id,device_id,transaction_timestamp,transaction_type,transaction_amount,payment_method,ip_address,...,unusual_location_flag,typing_speed_flag,shared_device_mule_count,known_fraud_ring_edge,biometric_anomaly_detected,automation_script_suspected,attack_vector_type,is_fraud,risk_score,risk_category
0,TXN_B0F9C19F0504,CUST_12184,BEN_21199,MER_146,DEV_00749,2026-04-27T03:38:19.906560Z,TRANSFER,381.05,NET_BANKING,192.168.82.203,...,0,0,1,0,0,0,NONE,0,32.96,LOW
1,TXN_D378C134E966,CUST_10447,BEN_20816,MER_162,DEV_00871,2026-04-27T03:38:58.906560Z,TRANSFER,488.25,WALLET,192.168.52.206,...,0,0,1,0,0,0,NONE,0,21.50,MEDIUM
2,TXN_D697BE131377,CUST_11009,BEN_21391,MER_080,DEV_00350,2026-04-27T03:39:37.906560Z,WITHDRAWAL,688.37,UPI,192.168.34.206,...,0,0,1,0,0,0,NONE,0,38.90,LOW
3,TXN_A711752E5B76,CUST_10070,BEN_20214,MER_050,DEV_00403,2026-04-27T03:40:16.906560Z,WITHDRAWAL,11490.06,CREDIT_CARD,155.166.212.59,...,1,0,12,1,1,1,MULE_RING,1,91.52,HIGH
4,TXN_6B0E58B96DD4,CUST_10206,BEN_21195,MER_180,DEV_01013,2026-04-27T03:40:55.906560Z,WITHDRAWAL,1647.05,NET_BANKING,192.168.63.139,...,0,0,1,0,0,0,NONE,0,38.20,LOW
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,TXN_1AA22FB3EA72,CUST_12459,BEN_21015,MER_085,DEV_00526,2026-06-11T06:55:04.906560Z,PAYMENT,812.78,NET_BANKING,192.168.94.148,...,0,0,1,0,0,0,NONE,0,32.21,LOW
99996,TXN_A457445B7418,CUST_11018,BEN_20273,MER_196,DEV_00096,2026-06-11T06:55:43.906560Z,PAYMENT,364.76,NET_BANKING,192.168.4.184,...,0,0,1,0,0,0,NONE,0,25.19,LOW
99997,TXN_C79735C1B835,CUST_12106,BEN_21039,MER_058,DEV_00734,2026-06-11T06:56:22.906560Z,PAYMENT,1171.90,CREDIT_CARD,192.168.88.186,...,0,0,1,0,0,0,NONE,0,25.07,MEDIUM
99998,TXN_48A35D400126,CUST_10301,BEN_21261,MER_052,DEV_00563,2026-06-11T06:57:01.906560Z,WITHDRAWAL,384.86,CREDIT_CARD,192.168.80.20,...,0,0,1,0,0,0,NONE,0,15.80,MEDIUM


In [ ]:
!pip install matplotlib

In [ ]:
# !pip install xgboost lightgbm scikit-learn pandas numpy matplotlib

import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, roc_auc_score, mean_absolute_error, r2_score

import xgboost as xgb
import lightgbm as lgb
import joblib

print("✅ Core ML transformer engines loaded.")

✅ Core ML transformer engines loaded.


In [13]:
DATA_PATH = "scaled_fraud_training_data_100k.csv"
if not os.path.exists(DATA_PATH):
    raise FileNotFoundError("Missing dataset source! Please run the 100k generation script first.")

df = pd.read_csv(DATA_PATH)

# Define feature spaces based on required mathematical treatments
log_features = ["transaction_amount", "avg_transaction_amount_7d"]
cyclical_features = ["hour_of_day"]
numeric_features = ["account_age_days", "transaction_frequency_24h", "failed_transaction_count_24h", "session_duration_minutes", "device_risk_score", "shared_device_mule_count"]
passthrough_features = ["is_international", "unusual_amount_flag", "unusual_location_flag", "typing_speed_flag", "known_fraud_ring_edge", "biometric_anomaly_detected", "automation_script_suspected"]

all_features = log_features + cyclical_features + numeric_features + passthrough_features
X = df[all_features]
y_clf = df["is_fraud"]
y_reg = df["risk_score"]

# Split into train/test splits before transformations to maintain pure validation environments
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X, y_clf, test_size=0.2, stratify=y_clf, random_state=42)
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y_reg, test_size=0.2, random_state=42)

print(f"📈 Splits initialized. Base feature matrix columns shape: {X.shape[1]}")

📈 Splits initialized. Base feature matrix columns shape: 16


In [14]:
def log_transform(x):
    return np.log1p(x)

def cyclical_hour_sine(x):
    return np.sin(2 * np.pi * x / 24.0)

def cyclical_hour_cosine(x):
    return np.cos(2 * np.pi * x / 24.0)

# Build custom Scikit-Learn compatible transformers
log_transformer = FunctionTransformer(log_transform, validate=True)
sin_transformer = FunctionTransformer(cyclical_hour_sine, validate=True)
cos_transformer = FunctionTransformer(cyclical_hour_cosine, validate=True)

print("🛠️ Custom mathematical transformations engineered.")

🛠️ Custom mathematical transformations engineered.


In [11]:
!pip install xgboost lightgbm joblib scikit-learn pandas numpy

   ---------------------------------------- 0.0/101.7 MB ? eta -:--:--
   - -------------------------------------- 2.9/101.7 MB 19.5 MB/s eta 0:00:06
   -- ------------------------------------- 6.6/101.7 MB 19.3 MB/s eta 0:00:05
   ---- ----------------------------------- 10.7/101.7 MB 19.3 MB/s eta 0:00:05
   ----- ---------------------------------- 13.6/101.7 MB 17.8 MB/s eta 0:00:05
   ------ --------------------------------- 16.5/101.7 MB 16.8 MB/s eta 0:00:06
   ------- -------------------------------- 19.4/101.7 MB 16.6 MB/s eta 0:00:05
   -------- ------------------------------- 22.5/101.7 MB 16.2 MB/s eta 0:00:05
   --------- ------------------------------ 23.9/101.7 MB 14.8 MB/s eta 0:00:06
   ---------- ----------------------------- 26.7/101.7 MB 14.8 MB/s eta 0:00:06
   ----------- ---------------------------- 29.6/101.7 MB 14.7 MB/s eta 0:00:05
   ------------ --------------------------- 32.5/101.7 MB 14.6 MB/s eta 0:00:05
   ------------- -------------------------- 34.1/10